In [ ]:
import os
import json
import time
import warnings
from datetime import datetime
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy import stats

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

def get_paths():
    try:
        import google.colab
        from google.colab import drive
        if not os.path.exists('/content/drive'):
            drive.mount('/content/drive')
        base = '/content/drive/My Drive'
    except ImportError:
        base = '/Users/klarameyer/Library/CloudStorage/GoogleDrive-klaradynm@gmail.com/My Drive'
    
    return {
        'base': base,
        '2comp': f'{base}/newthermochemistrydatasets/2-component-system',
        '3comp': f'{base}/newthermochemistrydatasets/3-component-system',
        '4comp': f'{base}/newthermochemistrydatasets/4-component-system',
        'output': f'{base}/thermochemistry_experiments/comprehensive_benchmarks'
    }

PATHS = get_paths()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

MASTER_ELEMENTS = ['Al', 'Ca', 'Mg', 'O', 'Si']

# All systems to benchmark
BINARY_SYSTEMS = {
    'al2o3-sio2-data.json': ['Al2O3', 'SiO2'],
    'cao-al2o3-data.json': ['CaO', 'Al2O3'],
    'cao-mgo-data.json': ['CaO', 'MgO'],
    'cao-sio2-data.json': ['CaO', 'SiO2'],
    'mgo-al2o3-data.json': ['MgO', 'Al2O3'],
    'mgo-sio2-data.json': ['MgO', 'SiO2'],
}

TERNARY_SYSTEMS = {
    'cao-al2o3-sio2-data.json': ['CaO', 'Al2O3', 'SiO2'],
    'cao-mgo-al2o3-formatted-data.json': ['CaO', 'MgO', 'Al2O3'],
    'cao-mgo-sio2-data.json': ['CaO', 'MgO', 'SiO2'],
    'mgo-al2o3-sio2-data.json': ['MgO', 'Al2O3', 'SiO2'],
}

QUATERNARY_SYSTEMS = {
    'cao-mgo-al2o3-sio2-formatted-data.json': ['CaO', 'MgO', 'Al2O3', 'SiO2'],
}


class Config:
    """Best configuration (Option F)"""
    phase_embed_dim = 256
    phase_num_heads = 8
    phase_num_layers = 3
    dropout = 0.1
    stage1_epochs = 60
    stage2_epochs = 100
    stage1_lr = 1e-3
    stage2_lr = 5e-4
    batch_size = 512
    stage1_patience = 20
    stage2_patience = 25
    lr_factor = 0.5
    lr_patience = 5
    min_lr = 1e-6
    presence_weight = 10.0
    fraction_weight = 10.0
    prop_weights = {'H': 5.0, 'S': 1.0, 'Cp': 10.0, 'G': 1.0}
    n_runs = 3  # Number of runs per system for statistical robustness


# ============================================================================
# DATA LOADING
# ============================================================================

def load_system_data(filepath):
    """Load data from a single system file."""
    if not os.path.exists(filepath):
        print(f"  WARNING: File not found: {filepath}")
        return None
    
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    samples = list(data.values()) if isinstance(data, dict) else data
    
    # Discover phases
    all_phases = set()
    for s in samples:
        for phase in s.get('phs', {}).keys():
            all_phases.add(phase)
    phases = sorted(list(all_phases))
    n_phases = len(phases)
    phase_idx = {p: i for i, p in enumerate(phases)}
    
    X, T_raw, presence, fractions, properties = [], [], [], [], []
    
    for s in samples:
        init = s.get('init', {})
        sc = init.get('sc', {})
        
        comp = [sc.get(e, 0.0) for e in MASTER_ELEMENTS]
        X.append(comp + [init.get('P', 1.0)])
        T_raw.append(init.get('T', 1500.0))
        
        pres = np.zeros(n_phases)
        frac = np.zeros(n_phases)
        prop = np.zeros((n_phases, 4))
        
        for phase_name, phase_data in s.get('phs', {}).items():
            if phase_name in phase_idx:
                i = phase_idx[phase_name]
                pres[i] = 1.0
                frac[i] = phase_data.get('X', 0.0)
                props = phase_data.get('properties', {})
                prop[i] = [props.get('H', 0), props.get('S', 0),
                          props.get('Cp', 0), props.get('G', 0)]
        
        presence.append(pres)
        fractions.append(frac)
        properties.append(prop)
    
    X = np.array(X, dtype=np.float32)
    T_raw = np.array(T_raw, dtype=np.float32)
    presence = np.array(presence, dtype=np.float32)
    fractions = np.array(fractions, dtype=np.float32)
    properties = np.array(properties, dtype=np.float32)
    
    # Outlier removal on Cp (only present phases)
    outlier_mask = np.zeros(len(X), dtype=bool)
    for pi in range(n_phases):
        phase_present = presence[:, pi] > 0.5
        if phase_present.sum() < 10:
            continue
        cp_data = properties[phase_present, pi, 2]
        z = np.abs(stats.zscore(cp_data))
        present_indices = np.where(phase_present)[0]
        for idx, z_val in zip(present_indices, z):
            if z_val > 3.0:
                outlier_mask[idx] = True
    
    clean = ~outlier_mask
    
    return {
        'X': X[clean], 'T': T_raw[clean],
        'presence': presence[clean], 'fractions': fractions[clean],
        'properties': properties[clean],
        'n_phases': n_phases, 'phases': phases,
        'n_outliers': outlier_mask.sum()
    }


def add_physics_features(X, T):
    """Add physics-guided features (31 total)."""
    R = 8.314
    n_comp = len(MASTER_ELEMENTS)
    compositions = X[:, :n_comp]
    T = T.reshape(-1, 1)
    
    features = [X, T, T**2, T**3, R*T, R*T**2, 1/(T+1e-6), 1/(T**2+1e-6)]
    
    # Pairwise interactions
    for i in range(n_comp):
        for j in range(i + 1, n_comp):
            features.append(compositions[:, i:i+1] * compositions[:, j:j+1])
    
    # Composition statistics
    comp_var = np.var(compositions, axis=1, keepdims=True)
    features.append(comp_var)
    
    # Entropy of mixing
    x_safe = compositions + 1e-10
    entropy_mix = -np.sum(compositions * np.log(x_safe), axis=1, keepdims=True)
    features.append(entropy_mix)
    features.append(T * entropy_mix)
    features.append(T * compositions)
    
    return np.hstack(features)


# ============================================================================
# DATASET
# ============================================================================

class ThermoDataset(Dataset):
    def __init__(self, X, T, presence, fractions, properties,
                 X_sc=None, prop_scalers=None, fit=False):
        
        self.T = torch.FloatTensor(T)
        self.presence = torch.FloatTensor(presence)
        self.fractions = torch.FloatTensor(fractions)
        self.properties_orig = torch.FloatTensor(properties)
        
        X_enh = add_physics_features(X, T)
        
        if fit:
            self.X_sc = StandardScaler()
            X_scaled = self.X_sc.fit_transform(X_enh)
        else:
            self.X_sc = X_sc
            X_scaled = self.X_sc.transform(X_enh)
        self.X = torch.FloatTensor(X_scaled)
        
        n_samples, n_phases, _ = properties.shape
        properties_scaled = np.zeros_like(properties)
        
        if fit:
            self.prop_scalers = {}
            for pi, pn in enumerate(['H', 'S', 'Cp', 'G']):
                scaler = StandardScaler()
                present_mask = presence > 0.5
                present_vals = properties[:, :, pi][present_mask]
                
                if len(present_vals) > 0:
                    scaler.fit(present_vals.reshape(-1, 1))
                else:
                    scaler.fit(properties[:, :, pi].reshape(-1, 1))
                
                properties_scaled[:, :, pi] = scaler.transform(
                    properties[:, :, pi].reshape(-1, 1)
                ).reshape(n_samples, n_phases)
                
                self.prop_scalers[pn] = scaler
        else:
            self.prop_scalers = prop_scalers
            for pi, pn in enumerate(['H', 'S', 'Cp', 'G']):
                properties_scaled[:, :, pi] = self.prop_scalers[pn].transform(
                    properties[:, :, pi].reshape(-1, 1)
                ).reshape(n_samples, n_phases)
        
        self.properties = torch.FloatTensor(properties_scaled)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return {
            'X': self.X[idx], 'T': self.T[idx],
            'pres': self.presence[idx], 'frac': self.fractions[idx],
            'prop': self.properties[idx], 'prop_orig': self.properties_orig[idx]
        }


# ============================================================================
# MODEL
# ============================================================================

class TransformerPhaseNet(nn.Module):
    def __init__(self, input_dim, n_phases, hidden_dim=256, nhead=8, num_layers=3):
        super().__init__()
        self.n_phases = n_phases
        
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.phase_queries = nn.Parameter(torch.randn(1, n_phases, hidden_dim))
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=nhead, dim_feedforward=hidden_dim*4,
            dropout=0.1, batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.presence_head = nn.Linear(hidden_dim, 1)
        self.fractions_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.GELU(),
            nn.Linear(hidden_dim // 2, 1)
        )
    
    def forward(self, x):
        batch_size = x.shape[0]
        x_proj = self.input_proj(x).unsqueeze(1)
        queries = self.phase_queries.expand(batch_size, -1, -1)
        seq = torch.cat([x_proj, queries], dim=1)
        encoded = self.transformer(seq)
        phase_repr = encoded[:, 1:, :]
        
        pres_logits = self.presence_head(phase_repr).squeeze(-1)
        presence = torch.sigmoid(pres_logits)
        
        frac_logits = self.fractions_head(phase_repr).squeeze(-1)
        mask = (presence > 0.5).float()
        masked_logits = frac_logits * mask + (-1e9) * (1 - mask)
        fractions = F.softmax(masked_logits, dim=1)
        
        return pres_logits, presence, fractions, phase_repr


class PropertyNet(nn.Module):
    def __init__(self, input_dim, n_phases, phase_hidden_dim=256):
        super().__init__()
        prop_input = input_dim + phase_hidden_dim * n_phases + n_phases * 2
        
        self.net = nn.Sequential(
            nn.Linear(prop_input, 512), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(512, 256), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(256, 128), nn.GELU(),
            nn.Linear(128, n_phases * 3)
        )
        self.n_phases = n_phases
    
    def forward(self, x, phase_repr, presence, fractions):
        batch_size = x.shape[0]
        phase_flat = phase_repr.reshape(batch_size, -1)
        augmented = torch.cat([x, phase_flat, presence, fractions], dim=1)
        output = self.net(augmented)
        return output.reshape(batch_size, self.n_phases, 3)


class TwoStageModel(nn.Module):
    def __init__(self, input_dim, n_phases):
        super().__init__()
        self.n_phases = n_phases
        
        self.register_buffer('H_mean', torch.tensor(0.0))
        self.register_buffer('H_std', torch.tensor(1.0))
        self.register_buffer('S_mean', torch.tensor(0.0))
        self.register_buffer('S_std', torch.tensor(1.0))
        self.register_buffer('G_mean', torch.tensor(0.0))
        self.register_buffer('G_std', torch.tensor(1.0))
        
        self.phase_net = TransformerPhaseNet(
            input_dim, n_phases, Config.phase_embed_dim,
            Config.phase_num_heads, Config.phase_num_layers
        )
        self.prop_net = PropertyNet(input_dim, n_phases, Config.phase_embed_dim)
    
    def set_scalers(self, prop_scalers):
        self.H_mean.fill_(prop_scalers['H'].mean_[0])
        self.H_std.fill_(prop_scalers['H'].scale_[0])
        self.S_mean.fill_(prop_scalers['S'].mean_[0])
        self.S_std.fill_(prop_scalers['S'].scale_[0])
        self.G_mean.fill_(prop_scalers['G'].mean_[0])
        self.G_std.fill_(prop_scalers['G'].scale_[0])
    
    def forward(self, x, temperature):
        pres_logits, presence, fractions, phase_repr = self.phase_net(x)
        hsc = self.prop_net(x, phase_repr, presence, fractions)
        H_norm, S_norm, Cp_norm = hsc[:, :, 0], hsc[:, :, 1], hsc[:, :, 2]
        
        # Hard constraint: G = H - TS
        H_orig = H_norm * self.H_std + self.H_mean
        S_orig = S_norm * self.S_std + self.S_mean
        T_exp = temperature.unsqueeze(1).expand(-1, self.n_phases)
        G_orig = H_orig - T_exp * S_orig
        G_norm = (G_orig - self.G_mean) / self.G_std
        
        properties = torch.stack([H_norm, S_norm, Cp_norm, G_norm], dim=2)
        
        return {
            'pres_logits': pres_logits, 'pres': presence,
            'frac': fractions, 'prop': properties
        }


# ============================================================================
# TRAINING
# ============================================================================

def train_model(model, train_loader, val_loader, device, verbose=True):
    """Train with masked loss."""
    
    # Stage 1: Phase prediction
    for p in model.prop_net.parameters():
        p.requires_grad = False
    
    optimizer = AdamW(model.phase_net.parameters(), lr=Config.stage1_lr, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, 'min', Config.lr_factor, Config.lr_patience, min_lr=Config.min_lr)
    
    best_loss, patience_cnt, best_state = float('inf'), 0, None
    
    for epoch in range(Config.stage1_epochs):
        model.train()
        for batch in train_loader:
            X, T = batch['X'].to(device), batch['T'].to(device)
            pres_t, frac_t = batch['pres'].to(device), batch['frac'].to(device)
            
            optimizer.zero_grad()
            out = model(X, T)
            loss = (Config.presence_weight * F.binary_cross_entropy_with_logits(out['pres_logits'], pres_t) +
                    Config.fraction_weight * F.mse_loss(out['frac'], frac_t))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.phase_net.parameters(), 1.0)
            optimizer.step()
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                X, T = batch['X'].to(device), batch['T'].to(device)
                pres_t, frac_t = batch['pres'].to(device), batch['frac'].to(device)
                out = model(X, T)
                val_loss += (F.binary_cross_entropy_with_logits(out['pres_logits'], pres_t) +
                            F.mse_loss(out['frac'], frac_t)).item()
        val_loss /= len(val_loader)
        scheduler.step(val_loss)
        
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= Config.stage1_patience:
                if verbose:
                    print(f"    S1 early stop ep {epoch+1}")
                break
        
        if verbose and (epoch + 1) % 20 == 0:
            print(f"    S1 Ep {epoch+1}: val_loss={val_loss:.4f}")
    
    if best_state:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    
    # Stage 2: Property prediction with masked loss
    for p in model.prop_net.parameters():
        p.requires_grad = True
    for p in model.phase_net.parameters():
        p.requires_grad = False
    
    optimizer = AdamW(model.prop_net.parameters(), lr=Config.stage2_lr, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, 'min', Config.lr_factor, Config.lr_patience, min_lr=Config.min_lr)
    
    best_loss, patience_cnt, best_state = float('inf'), 0, None
    
    for epoch in range(Config.stage2_epochs):
        model.train()
        for batch in train_loader:
            X, T = batch['X'].to(device), batch['T'].to(device)
            prop_t = batch['prop'].to(device)
            pres_t = batch['pres'].to(device)
            
            optimizer.zero_grad()
            out = model(X, T)
            
            mask = pres_t > 0.5
            
            if mask.sum() > 0:
                loss = 0
                for pi, (pn, weight) in enumerate(zip(['H', 'S', 'Cp', 'G'], 
                                                       [Config.prop_weights['H'], 
                                                        Config.prop_weights['S'],
                                                        Config.prop_weights['Cp'],
                                                        Config.prop_weights['G']])):
                    pred = out['prop'][:, :, pi][mask]
                    true = prop_t[:, :, pi][mask]
                    loss += weight * F.mse_loss(pred, true)
            else:
                loss = torch.tensor(0.0, device=device)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.prop_net.parameters(), 1.0)
            optimizer.step()
        
        model.eval()
        val_loss = 0
        val_count = 0
        with torch.no_grad():
            for batch in val_loader:
                X, T = batch['X'].to(device), batch['T'].to(device)
                prop_t = batch['prop'].to(device)
                pres_t = batch['pres'].to(device)
                out = model(X, T)
                
                mask = pres_t > 0.5
                if mask.sum() > 0:
                    for pi in range(4):
                        pred = out['prop'][:, :, pi][mask]
                        true = prop_t[:, :, pi][mask]
                        val_loss += F.mse_loss(pred, true).item()
                    val_count += 1
        
        if val_count > 0:
            val_loss /= val_count
        scheduler.step(val_loss)
        
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= Config.stage2_patience:
                if verbose:
                    print(f"    S2 early stop ep {epoch+1}")
                break
        
        if verbose and (epoch + 1) % 20 == 0:
            print(f"    S2 Ep {epoch+1}: val_loss={val_loss:.6f}")
    
    if best_state:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    
    for p in model.phase_net.parameters():
        p.requires_grad = True


# ============================================================================
# COMPREHENSIVE EVALUATION
# ============================================================================

def compute_all_metrics(model, test_loader, test_ds, device):
    """Compute all metrics needed for paper."""
    
    model.eval()
    
    all_pres_p, all_pres_t = [], []
    all_frac_p, all_frac_t = [], []
    all_prop_p, all_prop_t = [], []
    all_T = []
    
    with torch.no_grad():
        for batch in test_loader:
            X, T = batch['X'].to(device), batch['T'].to(device)
            out = model(X, T)
            
            all_pres_p.append(out['pres'].cpu().numpy())
            all_pres_t.append(batch['pres'].numpy())
            all_frac_p.append(out['frac'].cpu().numpy())
            all_frac_t.append(batch['frac'].numpy())
            all_prop_p.append(out['prop'].cpu().numpy())
            all_prop_t.append(batch['prop_orig'].numpy())
            all_T.append(batch['T'].numpy())
    
    pres_p = np.vstack(all_pres_p)
    pres_t = np.vstack(all_pres_t)
    frac_p = np.vstack(all_frac_p)
    frac_t = np.vstack(all_frac_t)
    prop_p_scaled = np.vstack(all_prop_p)
    prop_t = np.vstack(all_prop_t)
    T = np.concatenate(all_T)
    
    # Denormalize predictions
    n_phases = model.n_phases
    prop_p = np.zeros_like(prop_p_scaled)
    for pi, pn in enumerate(['H', 'S', 'Cp', 'G']):
        scaler = test_ds.prop_scalers[pn]
        prop_p[:, :, pi] = scaler.inverse_transform(
            prop_p_scaled[:, :, pi].reshape(-1, 1)
        ).reshape(-1, n_phases)
    
    results = {}
    
    # ===== Phase Metrics =====
    pres_binary_p = pres_p > 0.5
    pres_binary_t = pres_t > 0.5
    results['pres_acc'] = np.mean(pres_binary_p == pres_binary_t)
    
    # Per-phase metrics
    results['pres_precision'] = []
    results['pres_recall'] = []
    for pi in range(n_phases):
        tp = np.sum(pres_binary_p[:, pi] & pres_binary_t[:, pi])
        fp = np.sum(pres_binary_p[:, pi] & ~pres_binary_t[:, pi])
        fn = np.sum(~pres_binary_p[:, pi] & pres_binary_t[:, pi])
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        results['pres_precision'].append(precision)
        results['pres_recall'].append(recall)
    
    results['pres_precision_mean'] = np.mean(results['pres_precision'])
    results['pres_recall_mean'] = np.mean(results['pres_recall'])
    
    # Fraction MAE (only where phases are present)
    present_mask = pres_t > 0.5
    if present_mask.sum() > 0:
        results['frac_mae'] = np.mean(np.abs(frac_p[present_mask] - frac_t[present_mask]))
        results['frac_mae_pp'] = results['frac_mae'] * 100  # percentage points
    else:
        results['frac_mae'] = 0
        results['frac_mae_pp'] = 0
    
    # ===== Property Metrics =====
    for pi, pn in enumerate(['H', 'S', 'Cp', 'G']):
        pred = prop_p[:, :, pi][present_mask]
        true = prop_t[:, :, pi][present_mask]
        
        if len(true) > 0 and np.var(true) > 0:
            # R²
            ss_res = np.sum((true - pred) ** 2)
            ss_tot = np.sum((true - np.mean(true)) ** 2)
            results[f'{pn}_r2'] = 1 - ss_res / ss_tot
            
            # MAE
            results[f'{pn}_mae'] = np.mean(np.abs(true - pred))
            
            # MAPE (avoid division by zero)
            nonzero_mask = np.abs(true) > 1e-6
            if nonzero_mask.sum() > 0:
                results[f'{pn}_mape'] = np.mean(np.abs((true[nonzero_mask] - pred[nonzero_mask]) / true[nonzero_mask])) * 100
            else:
                results[f'{pn}_mape'] = float('nan')
            
            # RMSE
            results[f'{pn}_rmse'] = np.sqrt(np.mean((true - pred) ** 2))
            
            # NRMSE (normalized by range)
            data_range = np.max(true) - np.min(true)
            if data_range > 0:
                results[f'{pn}_nrmse'] = results[f'{pn}_rmse'] / data_range
            else:
                results[f'{pn}_nrmse'] = float('nan')
        else:
            results[f'{pn}_r2'] = float('nan')
            results[f'{pn}_mae'] = float('nan')
            results[f'{pn}_mape'] = float('nan')
            results[f'{pn}_rmse'] = float('nan')
            results[f'{pn}_nrmse'] = float('nan')
    
    # Overall R²
    valid_r2 = [results[f'{p}_r2'] for p in ['H', 'S', 'Cp', 'G'] 
                if not np.isnan(results[f'{p}_r2'])]
    results['overall_r2'] = np.mean(valid_r2) if valid_r2 else float('nan')
    
    # ===== Gibbs Consistency =====
    # Check G = H - TS
    H_pred = prop_p[:, :, 0][present_mask]
    S_pred = prop_p[:, :, 1][present_mask]
    G_pred = prop_p[:, :, 3][present_mask]
    T_expanded = np.repeat(T[:, np.newaxis], n_phases, axis=1)[present_mask]
    
    G_calculated = H_pred - T_expanded * S_pred
    results['gibbs_consistency_mae'] = np.mean(np.abs(G_pred - G_calculated))
    results['gibbs_consistency_max'] = np.max(np.abs(G_pred - G_calculated))
    
    return results


def benchmark_system(data_path, filename, system_type, seed=42):
    """Run full benchmark on a single system."""
    
    filepath = os.path.join(data_path, filename)
    data = load_system_data(filepath)
    
    if data is None:
        return None
    
    # Split data
    idx = np.arange(len(data['X']))
    train_idx, temp_idx = train_test_split(idx, test_size=0.3, random_state=seed)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=seed)
    
    train_ds = ThermoDataset(
        data['X'][train_idx], data['T'][train_idx],
        data['presence'][train_idx], data['fractions'][train_idx],
        data['properties'][train_idx], fit=True
    )
    val_ds = ThermoDataset(
        data['X'][val_idx], data['T'][val_idx],
        data['presence'][val_idx], data['fractions'][val_idx],
        data['properties'][val_idx],
        X_sc=train_ds.X_sc, prop_scalers=train_ds.prop_scalers
    )
    test_ds = ThermoDataset(
        data['X'][test_idx], data['T'][test_idx],
        data['presence'][test_idx], data['fractions'][test_idx],
        data['properties'][test_idx],
        X_sc=train_ds.X_sc, prop_scalers=train_ds.prop_scalers
    )
    
    train_loader = DataLoader(train_ds, batch_size=Config.batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=Config.batch_size)
    test_loader = DataLoader(test_ds, batch_size=Config.batch_size)
    
    input_dim = train_ds.X.shape[1]
    model = TwoStageModel(input_dim, data['n_phases']).to(DEVICE)
    model.set_scalers(train_ds.prop_scalers)
    
    # Train
    train_start = time.time()
    train_model(model, train_loader, val_loader, DEVICE, verbose=False)
    train_time = time.time() - train_start
    
    # Evaluate
    metrics = compute_all_metrics(model, test_loader, test_ds, DEVICE)
    metrics['train_time'] = train_time
    metrics['n_samples'] = len(data['X'])
    metrics['n_phases'] = data['n_phases']
    metrics['n_params'] = sum(p.numel() for p in model.parameters())
    
    # Inference speed
    model.eval()
    test_batch = next(iter(test_loader))
    X, T = test_batch['X'].to(DEVICE), test_batch['T'].to(DEVICE)
    
    # Warmup
    for _ in range(10):
        with torch.no_grad():
            _ = model(X, T)
    
    # Time inference
    n_iters = 100
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    start = time.time()
    for _ in range(n_iters):
        with torch.no_grad():
            _ = model(X, T)
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    inference_time = (time.time() - start) / n_iters
    
    metrics['inference_time_ms'] = inference_time * 1000
    metrics['inference_per_sample_us'] = (inference_time / len(X)) * 1e6
    
    return metrics


def run_benchmarks():
    """Run all benchmarks."""
    
    os.makedirs(PATHS['output'], exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    all_results = {
        'binary': {},
        'ternary': {},
        'quaternary': {},
        'summary': {}
    }
    
    print("=" * 80)
    print("COMPREHENSIVE BENCHMARKS FOR IJCNN PAPER")
    print("=" * 80)
    
    # ===== Binary Systems =====
    print("\n" + "=" * 60)
    print("BINARY SYSTEMS")
    print("=" * 60)
    
    binary_metrics = []
    
    for filename, components in BINARY_SYSTEMS.items():
        print(f"\n{filename}")
        print("-" * 40)
        
        run_results = []
        for run in range(Config.n_runs):
            seed = 42 + run
            print(f"  Run {run+1}/{Config.n_runs} (seed={seed})...", end=" ")
            
            metrics = benchmark_system(PATHS['2comp'], filename, 'binary', seed=seed)
            if metrics:
                run_results.append(metrics)
                print(f"R²={metrics['overall_r2']:.4f}")
        
        if run_results:
            # Aggregate across runs
            agg = aggregate_metrics(run_results)
            agg['system'] = filename
            agg['components'] = components
            all_results['binary'][filename] = agg
            binary_metrics.append(agg)
            
            print(f"\n  Summary: R²={agg['overall_r2_mean']:.4f}±{agg['overall_r2_std']:.4f}")
    
    # ===== Ternary Systems =====
    print("\n" + "=" * 60)
    print("TERNARY SYSTEMS")
    print("=" * 60)
    
    ternary_metrics = []
    
    for filename, components in TERNARY_SYSTEMS.items():
        print(f"\n{filename}")
        print("-" * 40)
        
        run_results = []
        for run in range(Config.n_runs):
            seed = 42 + run
            print(f"  Run {run+1}/{Config.n_runs} (seed={seed})...", end=" ")
            
            metrics = benchmark_system(PATHS['3comp'], filename, 'ternary', seed=seed)
            if metrics:
                run_results.append(metrics)
                print(f"R²={metrics['overall_r2']:.4f}")
        
        if run_results:
            agg = aggregate_metrics(run_results)
            agg['system'] = filename
            agg['components'] = components
            all_results['ternary'][filename] = agg
            ternary_metrics.append(agg)
            
            print(f"\n  Summary: R²={agg['overall_r2_mean']:.4f}±{agg['overall_r2_std']:.4f}")
    
    # ===== Quaternary System =====
    print("\n" + "=" * 60)
    print("QUATERNARY SYSTEM")
    print("=" * 60)
    
    for filename, components in QUATERNARY_SYSTEMS.items():
        print(f"\n{filename}")
        print("-" * 40)
        
        run_results = []
        for run in range(Config.n_runs):
            seed = 42 + run
            print(f"  Run {run+1}/{Config.n_runs} (seed={seed})...", end=" ")
            
            metrics = benchmark_system(PATHS['4comp'], filename, 'quaternary', seed=seed)
            if metrics:
                run_results.append(metrics)
                print(f"R²={metrics['overall_r2']:.4f}")
        
        if run_results:
            agg = aggregate_metrics(run_results)
            agg['system'] = filename
            agg['components'] = components
            all_results['quaternary'][filename] = agg
            
            print(f"\n  Summary: R²={agg['overall_r2_mean']:.4f}±{agg['overall_r2_std']:.4f}")
    
    # ===== Summary Tables =====
    print("\n" + "=" * 80)
    print("SUMMARY TABLES FOR PAPER")
    print("=" * 80)
    
    # Binary summary
    print("\n--- BINARY SYSTEMS ---")
    print(f"{'System':<25} {'H R²':>8} {'S R²':>8} {'Cp R²':>8} {'G R²':>8} {'Overall':>8} {'Pres Acc':>10} {'Frac MAE':>10}")
    print("-" * 95)
    for m in binary_metrics:
        print(f"{m['system']:<25} {m['H_r2_mean']:>8.4f} {m['S_r2_mean']:>8.4f} {m['Cp_r2_mean']:>8.4f} {m['G_r2_mean']:>8.4f} {m['overall_r2_mean']:>8.4f} {m['pres_acc_mean']*100:>9.1f}% {m['frac_mae_pp_mean']:>9.2f}pp")
    
    # Binary MAE/MAPE
    print(f"\n{'System':<25} {'H MAE':>12} {'S MAE':>12} {'Cp MAE':>12} {'G MAE':>12}")
    print("-" * 75)
    for m in binary_metrics:
        print(f"{m['system']:<25} {m['H_mae_mean']:>12.1f} {m['S_mae_mean']:>12.3f} {m['Cp_mae_mean']:>12.3f} {m['G_mae_mean']:>12.1f}")
    
    print(f"\n{'System':<25} {'H MAPE%':>10} {'S MAPE%':>10} {'Cp MAPE%':>10} {'G MAPE%':>10}")
    print("-" * 65)
    for m in binary_metrics:
        print(f"{m['system']:<25} {m['H_mape_mean']:>10.2f} {m['S_mape_mean']:>10.2f} {m['Cp_mape_mean']:>10.2f} {m['G_mape_mean']:>10.2f}")
    
    # Ternary summary
    print("\n--- TERNARY SYSTEMS ---")
    print(f"{'System':<35} {'H R²':>8} {'S R²':>8} {'Cp R²':>8} {'G R²':>8} {'Overall':>8} {'Pres Acc':>10}")
    print("-" * 95)
    for m in ternary_metrics:
        print(f"{m['system']:<35} {m['H_r2_mean']:>8.4f} {m['S_r2_mean']:>8.4f} {m['Cp_r2_mean']:>8.4f} {m['G_r2_mean']:>8.4f} {m['overall_r2_mean']:>8.4f} {m['pres_acc_mean']*100:>9.1f}%")
    
    # Ternary MAE/MAPE
    print(f"\n{'System':<35} {'H MAE':>12} {'S MAE':>12} {'Cp MAE':>12} {'G MAE':>12}")
    print("-" * 85)
    for m in ternary_metrics:
        print(f"{m['system']:<35} {m['H_mae_mean']:>12.1f} {m['S_mae_mean']:>12.3f} {m['Cp_mae_mean']:>12.3f} {m['G_mae_mean']:>12.1f}")
    
    print(f"\n{'System':<35} {'H MAPE%':>10} {'S MAPE%':>10} {'Cp MAPE%':>10} {'G MAPE%':>10}")
    print("-" * 75)
    for m in ternary_metrics:
        print(f"{m['system']:<35} {m['H_mape_mean']:>10.2f} {m['S_mape_mean']:>10.2f} {m['Cp_mape_mean']:>10.2f} {m['G_mape_mean']:>10.2f}")
    
    # Quaternary summary
    print("\n--- QUATERNARY SYSTEM ---")
    for filename, m in all_results['quaternary'].items():
        print(f"\nSystem: {filename}")
        print(f"  Samples: {m['n_samples_mean']:.0f}, Phases: {m['n_phases_mean']:.0f}")
        print(f"  Overall R²: {m['overall_r2_mean']:.4f} ± {m['overall_r2_std']:.4f}")
        print(f"  H  R²: {m['H_r2_mean']:.4f} ± {m['H_r2_std']:.4f}, MAE: {m['H_mae_mean']:.1f} J/mol, MAPE: {m['H_mape_mean']:.2f}%")
        print(f"  S  R²: {m['S_r2_mean']:.4f} ± {m['S_r2_std']:.4f}, MAE: {m['S_mae_mean']:.3f} J/(mol·K), MAPE: {m['S_mape_mean']:.2f}%")
        print(f"  Cp R²: {m['Cp_r2_mean']:.4f} ± {m['Cp_r2_std']:.4f}, MAE: {m['Cp_mae_mean']:.3f} J/(mol·K), MAPE: {m['Cp_mape_mean']:.2f}%")
        print(f"  G  R²: {m['G_r2_mean']:.4f} ± {m['G_r2_std']:.4f}, MAE: {m['G_mae_mean']:.1f} J/mol, MAPE: {m['G_mape_mean']:.2f}%")
        print(f"  Phase Presence Acc: {m['pres_acc_mean']*100:.1f}%")
        print(f"  Phase Fraction MAE: {m['frac_mae_pp_mean']:.2f} percentage points")
        print(f"  Gibbs Consistency MAE: {m['gibbs_consistency_mae_mean']:.6f} J/mol")
        print(f"  Training Time: {m['train_time_mean']:.1f}s")
        print(f"  Inference: {m['inference_time_ms_mean']:.2f} ms/batch, {m['inference_per_sample_us_mean']:.2f} µs/sample")
    
    # Save results
    def convert_for_json(obj):
        if isinstance(obj, (np.int64, np.int32)):
            return int(obj)
        elif isinstance(obj, (np.float64, np.float32)):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, dict):
            return {k: convert_for_json(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [convert_for_json(i) for i in obj]
        return obj
    
    with open(f'{PATHS["output"]}/comprehensive_benchmarks_{timestamp}.json', 'w') as f:
        json.dump(convert_for_json(all_results), f, indent=2)
    
    print(f"\n\nResults saved to: {PATHS['output']}")
    
    return all_results


def aggregate_metrics(run_results):
    """Aggregate metrics across multiple runs."""
    agg = {}
    
    # Get all keys from first result
    all_keys = run_results[0].keys()
    
    for key in all_keys:
        val = run_results[0][key]
        
        # Skip non-numeric types
        if isinstance(val, (list, dict, str, bool)):
            continue
        
        # Handle numeric types (int, float, numpy types)
        try:
            values = []
            for r in run_results:
                v = r[key]
                if v is not None and not (isinstance(v, float) and np.isnan(v)):
                    values.append(float(v))
            
            if values:
                agg[f'{key}_mean'] = np.mean(values)
                agg[f'{key}_std'] = np.std(values)
            else:
                agg[f'{key}_mean'] = float('nan')
                agg[f'{key}_std'] = float('nan')
        except (TypeError, ValueError):
            # Skip keys that can't be converted to float
            continue
    
    return agg


if __name__ == "__main__":
    run_benchmarks()